# Kicker Model - 01 Data Prep

This notebook centralizes data collection and feature engineering for NFL kicker modeling (seasons 2015-2024). It outputs modeling-ready CSVs in `Model/Data/` for downstream modeling work.


## Libraries


In [1]:
# Install only missing packages
pkgs <- c("tidyverse", "data.table", "nflreadr", "lubridate", "stringr",
      "forcats", "glue", "splines", "splines2", "readr")

installed <- rownames(installed.packages())
to_install <- setdiff(pkgs, installed)

if (length(to_install) > 0) {
  install.packages(to_install)
}

#Load libraries
suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(nflreadr)
  library(lubridate)
  library(stringr)
  library(forcats)
  library(glue)
  library(splines)
  library(splines2)
  library(readr)
})


## Configuration


In [6]:
#PROJECT_ROOT <- if (requireNamespace("here", quietly = TRUE)) here::here() else getwd()
PROJECT_ROOT <- "C:\\Users\\ekerv\\Proton Drive\\RPC1001 (1)\\My files\\Python\\Sports Analytics Projects\\Football\\Kickers"
DATA_DIR <- file.path(PROJECT_ROOT,"Model", "Data")
#dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)

SEASONS <- 2015:2024
options(tibble.print_min = 6, tibble.print_max = 6)

print(PROJECT_ROOT)
print(DATA_DIR)


[1] "C:\\Users\\ekerv\\Proton Drive\\RPC1001 (1)\\My files\\Python\\Sports Analytics Projects\\Football\\Kickers"
[1] "C:\\Users\\ekerv\\Proton Drive\\RPC1001 (1)\\My files\\Python\\Sports Analytics Projects\\Football\\Kickers/Model/Data"


## Load Play-by-Play Data



pbp_path <- file.path(DATA_DIR, "pbp_2000_2024.csv")

if (file.exists(pbp_path)) {
  pbp <- readr::read_csv(pbp_path, guess_max = 10000, show_col_types = FALSE)
} else {
  pbp <- nflreadr::load_pbp(seasons = SEASONS)
  readr::write_csv(pbp, pbp_path)
}

pbp <- pbp %>%
  mutate(
    season = as.integer(season),
    week = as.integer(week),
    qtr = as.integer(qtr),
    temp = suppressWarnings(as.numeric(temp)),
    wind = suppressWarnings(as.numeric(wind)),
    roof = forcats::fct_explicit_na(as.factor(roof), "unknown"),
    surface = forcats::fct_explicit_na(as.factor(surface), "unknown")
  )

pbp %>% count(season) %>% print(n = Inf)



In [ ]:
# Load data
pbp_path <- file.path(DATA_DIR, "pbp_2000_2024_clean.csv")
print(pbp_path)

#Load if file exits, if not return error
if (file.exists(pbp_path)) {
  pbp <- readr::read_csv(pbp_path, guess_max = 10000, show_col_types = FALSE)
} else {
  stop(glue("File {pbp_path} does not exist. Please download the data first."))
}

#Filter for 2015 - 2024
pbp <- pbp %>% filter(season %in% SEASONS)

[1] "C:\\Users\\ekerv\\Proton Drive\\RPC1001 (1)\\My files\\Python\\Sports Analytics Projects\\Football\\Kickers/Model/Data/pbp_2000_2024_clean.csv"


In [25]:
#Export pbp data to rds file, to "C:\Python\Data"
pbp %>%
  write_rds(file.path("C:\\Python\\Data", "pbp.rds"))

In [2]:
# Load pbp from RDS
pbp_rds_path <- "C:\\Python\\Data\\pbp.rds"

if (!file.exists(pbp_rds_path)) {
    stop(sprintf("File '%s' does not exist. Please run the cell that writes pbp.rds first.", pbp_rds_path))
}

pbp <- readr::read_rds(pbp_rds_path)

message(sprintf("Loaded pbp (%d rows, %d cols) from %s", nrow(pbp), ncol(pbp), pbp_rds_path))

# quick check
if ("season" %in% names(pbp)) {
    pbp %>% dplyr::count(season) %>% print(n = Inf)
} else {
    print(glue::glue("Loaded data does not contain a 'season' column."))
}

Loaded pbp (1184721 rows, 372 cols) from C:\Python\Data\pbp.rds



# A tibble: 25 × 2
   season     n
    <dbl> <int>
 1   2000 45491
 2   2001 44969
 3   2002 47355
 4   2003 46811
 5   2004 46705
 6   2005 46823
 7   2006 46299
 8   2007 46266
 9   2008 45917
10   2009 46519
11   2010 46892
12   2011 47448
13   2012 47834
14   2013 48158
15   2014 47629
16   2015 48122
17   2016 47651
18   2017 47245
19   2018 47109
20   2019 47260
21   2020 47705
22   2021 49922
23   2022 49434
24   2023 49665
25   2024 49492


In [7]:
print(head(pbp, 5))

# save first 5 rows of pbp to csv in DATA_DIR
readr::write_csv(head(pbp, 5), file.path(DATA_DIR, "pbp_head.csv"))

# A tibble: 5 × 372
  play_id game_id      old_game_id home_team away_team season_type  week posteam
    <dbl> <chr>              <dbl> <chr>     <chr>     <chr>       <dbl> <chr>  
1      34 2000_01_ARI…  2000090300 NYG       ARI       REG             1 ARI    
2      70 2000_01_ARI…  2000090300 NYG       ARI       REG             1 ARI    
3     106 2000_01_ARI…  2000090300 NYG       ARI       REG             1 ARI    
4     131 2000_01_ARI…  2000090300 NYG       ARI       REG             1 ARI    
5     148 2000_01_ARI…  2000090300 NYG       ARI       REG             1 ARI    
# ℹ 364 more variables: posteam_type <chr>, defteam <chr>, side_of_field <chr>,
#   yardline_100 <dbl>, game_date <date>, quarter_seconds_remaining <dbl>,
#   half_seconds_remaining <dbl>, game_seconds_remaining <dbl>,
#   game_half <chr>, quarter_end <dbl>, drive <dbl>, sp <dbl>, qtr <dbl>,
#   down <dbl>, goal_to_go <dbl>, time <time>, yrdln <chr>, ydstogo <dbl>,
#   ydsnet <dbl>, desc <chr>, play_type <chr>

In [26]:
# Find kicker_player_id for Justin Tucker (match common name variants)
justin_tucker_ids <- pbp %>%
    filter(!is.na(kicker_player_name) &
                 str_detect(kicker_player_name, regex("justin\\s*tucker|j\\.tucker|j\\s?tucker", ignore_case = TRUE))) %>%
    distinct(kicker_player_id, kicker_player_name) %>%
    arrange(kicker_player_id)

print(justin_tucker_ids)

# A tibble: 1 × 2
  kicker_player_id kicker_player_name
  <chr>            <chr>             
1 00-0029597       J.Tucker          


In [28]:
# Justin Tucker FG & XP attempts (2012-2019) with made/missed/blocked breakdown

seasons_jt <- 2012:2019

jt_kicks <- pbp %>%
  filter(
    season %in% seasons_jt,
    play_type %in% c("field_goal", "extra_point"),
    season_type == "REG",
    !is.na(kicker_player_name),
    str_detect(kicker_player_name, regex("justin\\s*tucker|j\\.tucker|j\\s?tucker", ignore_case = TRUE))
  ) %>%
  mutate(
    kick_result_raw = coalesce(field_goal_result, extra_point_result),
    kick_result = case_when(
      tolower(kick_result_raw) %in% c("made", "good") ~ "made",
      tolower(kick_result_raw) %in% c("missed", "failed") ~ "missed",
      tolower(kick_result_raw) %in% c("blocked") ~ "blocked",
      TRUE ~ NA_character_
    )
  )

# Breakdown tables
fg_summary <- jt_kicks %>%
  filter(play_type == "field_goal") %>%
  count(kick_result) %>%
  mutate(pct = n / sum(n))

xp_summary <- jt_kicks %>%
  filter(play_type == "extra_point") %>%
  count(kick_result) %>%
  mutate(pct = n / sum(n))

# Full detail (useful if you want the individual plays)
jt_detail <- jt_kicks %>%
  arrange(season, game_date, play_id) %>%
  select(season, game_date, week, game_id, play_id, play_type, kick_distance,
         kicker_player_name, kick_result_raw, kick_result, home_team, away_team)

# Print results
message("Justin Tucker: field goals (2012-2019)")
print(fg_summary)
message("Justin Tucker: extra points (2012-2019)")
print(xp_summary)

# Return invisibly for further use
invisible(list(summary_fg = fg_summary, summary_xp = xp_summary, detail = jt_detail))
unique(jt_detail$season)

Justin Tucker: field goals (2012-2019)



# A tibble: 3 × 3
  kick_result     n    pct
  <chr>       <int>  <dbl>
1 blocked         5 0.0171
2 made          265 0.908 
3 missed         22 0.0753


Justin Tucker: extra points (2012-2019)



# A tibble: 2 × 3
  kick_result     n     pct
  <chr>       <int>   <dbl>
1 made          298 0.990  
2 missed          3 0.00997


[1] 2012 2013 2014 2015 2016 2017 2018 2019

In [7]:
#Find all cols with wp in the title
wp_cols <- grep("post", names(pbp), value = TRUE)
print(wp_cols)

[1] "posteam"                    "posteam_type"              
[3] "posteam_timeouts_remaining" "posteam_score"             
[5] "posteam_score_post"         "defteam_score_post"        
[7] "score_differential_post"    "home_wp_post"              
[9] "away_wp_post"              


## Previous Play Features (lag within game)


In [5]:
pbp_prev <- pbp %>%
  arrange(game_id, play_id) %>%
  group_by(game_id) %>%
  mutate(
    prev_play_type = dplyr::lag(play_type),
    prev_desc = dplyr::lag(desc),
    prev_timeout = as.integer(dplyr::lag(timeout)),
    prev_timeout_team = dplyr::lag(timeout_team),
    prev_penalty = as.integer(dplyr::lag(penalty)),
    prev_incomplete = as.integer(dplyr::lag(incomplete_pass)),
    prev_out_bounds = as.integer(dplyr::lag(out_of_bounds)),
    prev_gsr = dplyr::lag(game_seconds_remaining),
    delta_secs = prev_gsr - game_seconds_remaining,
    prev_end_quarter = as.integer(!is.na(prev_desc) & str_detect(prev_desc, "(?i)end\\s+quarter")),
    prev_two_min_warning = as.integer(!is.na(prev_desc) & str_detect(prev_desc, "(?i)two-?minute\\s+warning"))
  ) %>%
  ungroup() %>%
  select(
    game_id, play_id,
    prev_play_type, prev_desc, prev_timeout, prev_timeout_team,
    prev_penalty, prev_incomplete, prev_out_bounds,
    prev_end_quarter, prev_two_min_warning, delta_secs
  )


## Field Goal & PAT Attempts


In [62]:
fg_attempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    play_type %in% c("field_goal", "extra_point"),
    (field_goal_result %in% c("made", "missed", "blocked")) |
      (extra_point_result %in% c("good", "failed", "blocked"))
  ) %>%
  mutate(
    is_pat = as.integer(play_type == "extra_point"),
    kick_result_raw = dplyr::coalesce(field_goal_result, extra_point_result),
    kick_result = case_when(
      kick_result_raw %in% c("made", "good") ~ "made",
      kick_result_raw %in% c("missed", "failed") ~ "missed",
      kick_result_raw %in% c("blocked") ~ "blocked",
      TRUE ~ NA_character_
    ),
    kick_distance = suppressWarnings(as.numeric(kick_distance)),
    kick_distance = if_else(is_pat == 1L & is.na(kick_distance), 33, kick_distance),
    attempted = 1L
  ) %>%
  filter(!is.na(kick_distance)) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season),
    week = as.integer(week),
    season_type,
    playoffs = as.integer(season_type == "POST"),
    qtr = as.integer(qtr),
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, home_wp, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go, 
    is_ot  = qtr >= 5,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance, kicker_player_id, kicker_player_name,
    field_goal_result, extra_point_result,
    kick_result, is_pat, attempted,
    play_type_original = play_type
  )


## Fourth-Quarter Non-Attempt Opportunities


In [8]:
fg_nonattempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    qtr == 4L,
    down == 4L,
    !is.na(yardline_100),
    !is.na(ydstogo)
  ) %>%
  mutate(
    derived_kick_distance = yardline_100 + 17L,   # LOS + 17 yards (7 hold + 10 EZ)
    fg_50_or_less = derived_kick_distance <= 50,
    one_score_tight = dplyr::between(score_differential, -3, 5),
    two_plus = ydstogo >= 2
  ) %>%
  # Keep only **non-kick** plays: offense play, punt, kneel/spike, or an explicit fake FG
  filter(
    fg_50_or_less, one_score_tight, two_plus,
    play_type %in% c("pass","run","punt","qb_kneel","qb_spike") |
      (special_teams_play == 1 & field_goal_attempt == 0 &
         play_type %in% c("pass","run"))   # fakes
  ) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season),
    week = as.integer(week),
    season_type,
    playoffs = as.integer(season_type == "POST"),
    qtr = as.integer(qtr), down,
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, home_wp, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance = derived_kick_distance,
    kicker_player_id = NA_character_,
    kicker_player_name = NA_character_,
    field_goal_result = NA_character_,
    extra_point_result = NA_character_,
    kick_result = NA_character_,
    is_pat = 0L,
    is_ot  = qtr >= 5,
    attempted = 0L,
    play_type_original = play_type
  )


In [10]:
#Export fg_nonattempts_raw to csv in DATA_DIR
fg_nonattempts_raw %>%
  write_csv(file.path(DATA_DIR, "fg_nonattempts_raw.csv"))

In [ ]:
# How many total eligible 4Q 4th-down plays inside 50 by play type?
pbp %>%
  filter(season %in% SEASONS, qtr == 4L, down == 4L,
         !is.na(yardline_100), !is.na(ydstogo)) %>%
  mutate(dist = yardline_100 + 17L) %>%
  filter(dplyr::between(score_differential, -10, 6),
         ydstogo >= 1, dist <= 50) %>%
  count(play_type, sort = TRUE)

# Of those, how many were actual kicks vs non-kicks?
pbp %>%
  filter(season %in% SEASONS, qtr == 4L, down == 4L,
         !is.na(yardline_100), !is.na(ydstogo)) %>%
  mutate(dist = yardline_100 + 17L,
         in_window = dplyr::between(score_differential, -10, 6) &
                     ydstogo >= 1 & dist <= 50) %>%
  summarise(
    total_window = sum(in_window),
    kicks = sum(in_window & play_type == "field_goal"),
    non_kicks = sum(in_window &
                    (play_type %in% c("pass","run","punt","qb_kneel","qb_spike") |
                       (special_teams_play == 1 & field_goal_attempt == 0 &
                          play_type %in% c("pass","run"))))
  )


play_type,n
<chr>,<int>
field_goal,1293
pass,320
run,146
no_play,106
punt,1
NA,1


total_window,kicks,non_kicks
<int>,<int>,<int>
1867,NA,467


## Combine Attempts & Opportunities for Shared Feature Engineering


In [64]:
fg_all <- bind_rows(fg_attempts_raw, fg_nonattempts_raw) %>%
  left_join(pbp_prev, by = c("game_id", "play_id"))


## Situational Features


In [65]:
fg_all <- fg_all %>%
  mutate(
    l2m = as.integer(qtr %in% c(2L, 4L) & quarter_seconds_remaining <= 120),
    clock_running = as.integer(
      l2m == 1L & is_pat == 0L &
      !is.na(delta_secs) & delta_secs > 0 &
      coalesce(prev_timeout, 0L) == 0L &
      coalesce(prev_penalty, 0L) == 0L &
      coalesce(prev_incomplete, 0L) == 0L &
      coalesce(prev_out_bounds, 0L) == 0L &
      coalesce(prev_end_quarter, 0L) == 0L &
      coalesce(prev_two_min_warning, 0L) == 0L
    ),
    iced = as.integer(
      is_pat == 0L &
      coalesce(prev_timeout, 0L) == 1L &
      !is.na(prev_timeout_team) &
      prev_timeout_team == defteam
    )
  )


## Kicker Information (age & experience)


In [66]:
players <- nflreadr::load_players() %>%
  transmute(
    gsis_id = as.character(gsis_id),
    birth_date = as.Date(birth_date),
    rookie_season = suppressWarnings(as.integer(rookie_season))
  )

fg_all <- fg_all %>%
  mutate(kicker_player_id = as.character(kicker_player_id)) %>%
  left_join(players, by = c("kicker_player_id" = "gsis_id")) %>%
  mutate(
    kicker_age = as.numeric(difftime(game_date, birth_date, units = "days")) / 365.25,
    kicker_experience = if_else(
      !is.na(rookie_season),
      as.numeric(season - rookie_season) + 1,
      NA_real_
    )
  )


## Venue Flags


In [67]:
fg_all <- fg_all %>%
  mutate(
    roof = forcats::fct_explicit_na(as.factor(roof), "unknown"),
    surface = forcats::fct_explicit_na(as.factor(surface), "unknown"),
    roof_std = tolower(as.character(roof)),
    indoors = as.integer(roof_std %in% c("dome", "closed", "open")),
    surface_std = tolower(as.character(surface)),
    is_turf = as.integer(surface_std != "grass"),
    high_altitude = as.integer(!is.na(stadium_id) & str_detect(stadium_id, "^(DEN|MEX)"))
  )


## Weather Parsing & Imputation


In [68]:
fg_all <- fg_all %>%
  mutate(
    weather = if_else(weather == "", NA_character_, weather),
    weather_first = if_else(
      is.na(weather),
      NA_character_,
      str_squish(str_to_lower(str_replace(weather, "(?i)\\s*temp:.*$", "")))
    ),
    weather_clean = weather_first,
    temp_from_weather = str_extract(weather, "(?i)temp\\s*:?\\s*(-?\\d{1,3})"),
    temp_from_weather = suppressWarnings(as.numeric(str_extract(temp_from_weather, "-?\\d{1,3}"))),
    temp_degree = str_extract(weather, "(?i)(-?\\d{1,3})\\s*(?:deg|degrees|\\u00B0)?\\s*f"),
    temp_degree = suppressWarnings(as.numeric(str_extract(temp_degree, "-?\\d{1,3}"))),
    wind_from_weather = str_extract(weather, "(?i)(\\d{1,3})\\s*mph"),
    wind_from_weather = suppressWarnings(as.numeric(str_extract(wind_from_weather, "\\d{1,3}"))),
    humidity_from_weather = str_extract(weather, "(?i)(\\d{1,3})\\s*%"),
    humidity_from_weather = suppressWarnings(as.numeric(str_extract(humidity_from_weather, "\\d{1,3}"))),
    temp = coalesce(temp, temp_from_weather, temp_degree),
    wind = coalesce(wind, wind_from_weather),
    humidity = humidity_from_weather
  ) %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 1L, 71, temp),
    wind = if_else(is.na(wind) & indoors == 1L, 0, wind),
    humidity = if_else(is.na(humidity) & indoors == 1L, 45, humidity)
  )

outdoor_monthly_medians <- fg_all %>%
  filter(indoors == 0L) %>%
  mutate(month = lubridate::month(game_date)) %>%
  group_by(stadium_id, month) %>%
  summarise(
    temp_median = suppressWarnings(as.numeric(median(temp, na.rm = TRUE))),
    wind_median = suppressWarnings(as.numeric(median(wind, na.rm = TRUE))),
    humidity_median = suppressWarnings(as.numeric(median(humidity, na.rm = TRUE))),
    .groups = "drop"
  )

fg_all <- fg_all %>%
  mutate(month = lubridate::month(game_date)) %>%
  left_join(outdoor_monthly_medians, by = c("stadium_id", "month")) %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 0L, temp_median, temp),
    wind = if_else(is.na(wind) & indoors == 0L, wind_median, wind),
    humidity = if_else(is.na(humidity) & indoors == 0L, humidity_median, humidity)
  )

global_temp_median <- fg_all %>% filter(indoors == 0L, !is.na(temp)) %>% summarise(median = median(temp)) %>% pull()
if (length(global_temp_median) == 0) global_temp_median <- NA_real_

global_wind_median <- fg_all %>% filter(indoors == 0L, !is.na(wind)) %>% summarise(median = median(wind)) %>% pull()
if (length(global_wind_median) == 0) global_wind_median <- NA_real_

global_humidity_median <- fg_all %>% filter(indoors == 0L, !is.na(humidity)) %>% summarise(median = median(humidity)) %>% pull()
if (length(global_humidity_median) == 0) global_humidity_median <- NA_real_

fg_all <- fg_all %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 0L, global_temp_median, temp),
    wind = if_else(is.na(wind) & indoors == 0L, global_wind_median, wind),
    humidity = if_else(is.na(humidity) & indoors == 0L, global_humidity_median, humidity)
  ) %>%
  select(-temp_from_weather, -temp_degree, -wind_from_weather, -humidity_from_weather,
         -temp_median, -wind_median, -humidity_median, -month)


## Weather Flags


In [69]:
fg_all <- fg_all %>%
  mutate(
    # normalize text early
    weather_clean = str_to_lower(coalesce(weather, "")) %>% str_squish(),
    weather_clean = if_else(weather_clean == "" & indoors == 0L, "clear", weather_clean),

    # fix common typos and variants
    weather_clean = str_replace_all(weather_clean, "(cloudly|coudy|cloundy|clo[iu]dy)", "cloudy"),
    weather_clean = str_replace_all(weather_clean, "(mosly|mostly\\s+coudy)", "mostly cloudy"),
    weather_clean = str_replace_all(weather_clean, "(partly\\s*sunny|sun\\s*/\\s*clouds|sun\\s*&\\s*clouds|sunny\\s*intervals)", "partly cloudy"),
    weather_clean = str_replace_all(weather_clean, "hazey", "hazy"),

    # detect explicit "no rain" phrases
    no_rain_phrase = as.integer(str_detect(weather_clean, "no\\s+chance\\s+of\\s+rain|0%\\s*chance\\s*of\\s+rain|zero\\s*percent\\s*chance\\s*of\\s+rain")),

    # weather flags (note double-escaped backslashes)
    is_snow_sleet = as.integer(
      indoors != 1L & str_detect(weather_clean, "\\bsnow\\b|\\bflurr(y|ies)\\b|\\bsleet\\b|\\bfreezing\\s+rain\\b|\\bwintry\\s+mix\\b|\\bice\\b")
    ),
    is_rain_showers = as.integer(
      indoors != 1L & str_detect(weather_clean, "\\brain\\b|\\braining\\b|\\bshowers?\\b|\\bdrizzle\\b|\\bstorm\\b|\\bthunderstorm\\b")
    ),
    # suppress rain if snow mentioned or explicit "no rain"
    is_rain_showers = if_else(is_snow_sleet == 1L | no_rain_phrase == 1L, 0L, is_rain_showers),

    is_cloudy = as.integer(
      indoors != 1L & str_detect(weather_clean, "\\bcloudy\\b|\\bovercast\\b|\\bpartly\\s+cloudy\\b|\\bmostly\\s+cloudy\\b|\\bscattered\\s+clouds?\\b")
    ),
    is_hazy_fog = as.integer(
      indoors != 1L & str_detect(weather_clean, "\\bfog(?:gy)?\\b|\\bhaze\\b|\\bhazy\\b|\\bmist\\b")
    )
  ) %>%
  select(-no_rain_phrase)


## Adding Leverage

In [70]:
# === Adding Leverage (uses artifacts from 01_leverage.ipynb) ===================

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(purrr)
})

# Project dirs (match 01_leverage.ipynb)
LEVERAGE_DIR <- file.path(PROJECT_ROOT, "Model", "Leverage")
dir.create(LEVERAGE_DIR, recursive = TRUE, showWarnings = FALSE)

# Load FG heads + rules
fg_pack_path  <- file.path(LEVERAGE_DIR, "wp_heads_rulepack.rds")
stopifnot(file.exists(fg_pack_path))
fg_pack <- readRDS(fg_pack_path)
mod_make        <- fg_pack$mod_make
mod_miss        <- fg_pack$mod_miss
mk_feats_make   <- fg_pack$mk_feats_make
mk_feats_miss   <- fg_pack$mk_feats_miss
apply_end_rules <- fg_pack$apply_endgame_rules

# Load PAT heads + rules
pat_pack_path <- file.path(LEVERAGE_DIR, "pat_heads_rulepack.rds")
stopifnot(file.exists(pat_pack_path))
pat_pack <- readRDS(pat_pack_path)
mod_pat_make      <- pat_pack$mod_pat_make
mod_pat_miss      <- pat_pack$mod_pat_miss
mk_feats_pat_make <- pat_pack$mk_feats_pat_make
mk_feats_pat_miss <- pat_pack$mk_feats_pat_miss
apply_pat_rules   <- pat_pack$apply_pat_rules_min
pat_ot_const      <- pat_pack$pat_ot_const

# Helper
eps <- 1e-6
clamp01 <- function(x) pmin(pmax(x, eps), 1 - eps)

# put this once near the top of the cell, after clamp01()
desired_cols <- c(
  ".row_id",
  "wp_make_hat", "wp_miss_hat", "leverage",
  "wp_make_hat_rules", "wp_miss_hat_rules",
  # some rulepacks include this; many don't — that's OK:
  "pred_hat_post",
  "leverage_rules"
)

# Row id to merge results back cleanly
fg_all <- fg_all %>% mutate(.row_id = dplyr::row_number())

# -------------------- FG (attempts + theoretical non-attempts) -----------------
fg_nonpat <- fg_all %>% filter(is_pat == 0L)

# Only score non-OT rows (mirrors the leverage notebook)
fg_nonpat_scorable <- fg_nonpat %>%
  filter(!is_ot) %>%
  # the miss head uses yardline_100 as a predictor in the leverage notebook;
  # keep only rows where that exists (mirrors training/valid prep)
  filter(!is.na(yardline_100))

if (nrow(fg_nonpat_scorable) > 0) {
  # Feature frames per head (must be built BEFORE prediction)
  feats_make <- mk_feats_make(fg_nonpat_scorable)
  feats_miss <- mk_feats_miss(fg_nonpat_scorable)

  scored_fg <- fg_nonpat_scorable %>%
    mutate(
      wp_make_hat = clamp01(predict(mod_make, newdata = feats_make, type = "response")),
      wp_miss_hat = clamp01(predict(mod_miss, newdata = feats_miss, type = "response")),
      leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
    ) %>%
    apply_end_rules() %>%
    dplyr::select(dplyr::any_of(desired_cols))
} else {
  scored_fg <- tibble(.row_id = integer(),
                      wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                      wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                      pred_hat_post = numeric(), leverage_rules = numeric())
}

# -------------------- FG (OT) — constant leverage like leverage notebook -------
# -------------------- FG (OT) — constant leverage (from late Q4 reg FG) -------
# OT FG rows we skipped earlier
fg_nonpat_ot <- fg_nonpat %>% filter(is_ot)

# Join leverage back onto the scorable REG rows so we can filter on Q4/time/score
const_pool <- fg_nonpat_scorable %>%
  inner_join(
    scored_fg %>% dplyr::select(.row_id, leverage_rules),
    by = ".row_id"
  ) %>%
  # Late Q4 tie/go-ahead cohort, matching leverage notebook logic
  dplyr::filter(
    qtr == 4L,
    quarter_seconds_remaining < 120,
    score_differential %in% -3:0
  )

fg_ot_const_val <- const_pool %>%
  summarise(med = median(leverage_rules, na.rm = TRUE)) %>%
  pull(med)

# Fallbacks if the cohort is empty
if (is.na(fg_ot_const_val)) {
  fg_ot_const_val <- scored_fg %>%
    summarise(med = median(leverage_rules, na.rm = TRUE)) %>% pull(med)
}
if (is.na(fg_ot_const_val)) fg_ot_const_val <- 0.5

# Materialize OT FG rows with the constant leverage values
if (nrow(fg_nonpat_ot) > 0) {
  scored_fg_ot <- fg_nonpat_ot %>%
    transmute(
      .row_id,
      wp_make_hat       = NA_real_,
      wp_miss_hat       = NA_real_,
      leverage          = fg_ot_const_val,
      wp_make_hat_rules = NA_real_,
      wp_miss_hat_rules = NA_real_,
      pred_hat_post     = NA_real_,
      leverage_rules    = fg_ot_const_val
    ) %>%
    dplyr::select(dplyr::any_of(desired_cols))
} else {
  scored_fg_ot <- tibble(.row_id = integer(),
                         wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                         wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                         pred_hat_post = numeric(), leverage_rules = numeric())
}

# ------------------------------- PATs ------------------------------------------
pats <- fg_all %>% filter(is_pat == 1L)

# Split OT vs regulation for PAT path (leverage notebook treats OT with a constant helper)
pats_nonot <- pats %>% filter(!is_ot)
pats_ot    <- pats %>% filter(is_ot)

if (nrow(pats_nonot) > 0) {
  feats_pat_make <- mk_feats_pat_make(pats_nonot)
  feats_pat_miss <- mk_feats_pat_miss(pats_nonot)

  # PAT non-OT path (replace the existing select(...) line)
  scored_pat_nonot <- pats_nonot %>%
    mutate(
      wp_make_hat = clamp01(predict(mod_pat_make, newdata = feats_pat_make, type = "response")),
      wp_miss_hat = clamp01(predict(mod_pat_miss, newdata = feats_pat_miss, type = "response")),
      leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
    ) %>%
    apply_pat_rules() %>%
    dplyr::select(dplyr::any_of(desired_cols))

} else {
  scored_pat_nonot <- tibble(.row_id = integer(),
                             wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                             wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                             pred_hat_post = numeric(), leverage_rules = numeric())
}

# PAT OT: use the constant helper saved by the leverage notebook
if (nrow(pats_ot) > 0) {
  scored_pat_ot <- pat_ot_const(pats_ot) %>%
    dplyr::select(dplyr::any_of(desired_cols))
} else {
  scored_pat_ot <- tibble(.row_id = integer(),
                          wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                          wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                          pred_hat_post = numeric(), leverage_rules = numeric())
}

# -------------------------- Bind & write back to fg_all ------------------------
scored_all <- bind_rows(scored_fg, scored_fg_ot, scored_pat_nonot, scored_pat_ot)

fg_all <- fg_all %>%
  left_join(scored_all, by = ".row_id") %>%
  select(-.row_id)

message(sprintf(
  "Leverage added: FG (reg) = %d, FG (OT const) = %d, PAT (reg) = %d, PAT (OT) = %d.  FG OT const = %.4f",
  nrow(scored_fg), nrow(scored_fg_ot), nrow(scored_pat_nonot), nrow(scored_pat_ot), fg_ot_const_val
))

# === Populate pred_hat_post from realized outcomes ============================

# Normalize common result strings (robust to different feeds)
fg_made_vals   <- c("made", "good")                # FG 'made' is usually "made"
fg_miss_vals   <- c("missed", "blocked")           # treat blocks as misses for WP
pat_good_vals  <- c("good")                        # PAT made
pat_fail_vals  <- c("failed", "missed", "blocked", "aborted", "no_good")

fg_all <- fg_all %>%
  mutate(
    # Prefer rule-adjusted heads (already clamped/rule-layered)
    pred_hat_post = dplyr::case_when(
      # Field goals with realized outcome
      is_pat == 0L & attempted == 1L & !is.na(kick_result) &
        tolower(kick_result) %in% fg_made_vals
          ~ wp_make_hat_rules,

      is_pat == 0L & attempted == 1L & !is.na(kick_result) &
        tolower(kick_result) %in% fg_miss_vals
          ~ wp_miss_hat_rules,

      # PATs with realized outcome
      is_pat == 1L & !is.na(extra_point_result) &
        tolower(extra_point_result) %in% pat_good_vals
          ~ wp_make_hat_rules,

      is_pat == 1L & !is.na(extra_point_result) &
        tolower(extra_point_result) %in% pat_fail_vals
          ~ wp_miss_hat_rules,

      # otherwise: keep existing value (e.g., if a rules fn already set it) or remain NA
      TRUE ~ pred_hat_post
    ),
    # Clamp for safety
    pred_hat_post = dplyr::if_else(
      !is.na(pred_hat_post),
      pmin(pmax(pred_hat_post, 0), 1),
      pred_hat_post
    )
  )

# Quick sanity: how many rows are now populated?
message(sprintf("pred_hat_post populated for %d rows (%.1f%%).",
                sum(!is.na(fg_all$pred_hat_post)),
                100*mean(!is.na(fg_all$pred_hat_post))))

Leverage added: FG (reg) = 10521, FG (OT const) = 138, PAT (reg) = 12749, PAT (OT) = 0.  FG OT const = 0.5462

pred_hat_post populated for 23235 rows (99.3%).



In [71]:
# === Sanity checks & completeness (robust to 0/1 flags) =======================

suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(purrr); library(glue)
})

assert_true <- function(ok, msg) if (!isTRUE(ok)) stop(msg, call. = FALSE)
assert_between01 <- function(x, allow_na = TRUE, label = deparse(substitute(x))) {
  if (allow_na) x <- x[!is.na(x)]
  bad <- any(x < 0 | x > 1)
  assert_true(!bad, glue("{label} must be within [0,1]."))
}

# Stable row id for diagnostics (not modifying fg_all permanently)
fg_chk <- fg_all %>% mutate(.row_id_check = dplyr::row_number())

# --- NEW: canonical logical flags ---------------------------------------------
# as.logical() handles 0/1 or TRUE/FALSE cleanly
fg_chk <- fg_chk %>%
  mutate(
    pat_flag = as.logical(is_pat),
    ot_flag  = as.logical(is_ot)
  )

# 1) Columns exist
req_cols <- c(
  "is_pat", "is_ot",
  "wp_make_hat", "wp_miss_hat", "leverage",
  "wp_make_hat_rules", "wp_miss_hat_rules", "leverage_rules"
)
missing_cols <- setdiff(req_cols, names(fg_chk))
assert_true(length(missing_cols) == 0,
            glue("Missing expected columns: {paste(missing_cols, collapse=', ')}"))

# 2) Basic ranges & consistency
assert_between01(fg_chk$wp_make_hat,       TRUE, "wp_make_hat")
assert_between01(fg_chk$wp_miss_hat,       TRUE, "wp_miss_hat")
assert_between01(fg_chk$leverage,          TRUE, "leverage")
assert_between01(fg_chk$wp_make_hat_rules, TRUE, "wp_make_hat_rules")
assert_between01(fg_chk$wp_miss_hat_rules, TRUE, "wp_miss_hat_rules")
assert_between01(fg_chk$leverage_rules,    TRUE, "leverage_rules")

fg_chk <- fg_chk %>%
  mutate(.lev_raw_from_heads = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1))
diff_eps <- with(fg_chk, abs(leverage - .lev_raw_from_heads))
if (any(!is.na(diff_eps))) {
  assert_true(max(diff_eps, na.rm = TRUE) < 1e-6,
              "leverage != clamp(wp_make_hat - wp_miss_hat) on some rows.")
}

# 3) Coverage rules by play type & period
fg_nonpat <- fg_chk %>% filter(!pat_flag)
fg_pats   <- fg_chk %>% filter( pat_flag)

fg_nonpat_nonot <- fg_nonpat %>% filter(!ot_flag)
fg_nonpat_nonot_na <- fg_nonpat_nonot %>%
  summarise(
    n = n(),
    na_wp_make = sum(is.na(wp_make_hat)),
    na_wp_miss = sum(is.na(wp_miss_hat)),
    na_lev     = sum(is.na(leverage))
  )

fg_nonpat_ot <- fg_nonpat %>% filter(ot_flag)

pats_na <- fg_pats %>%
  summarise(
    n = n(),
    na_wp_make    = sum(is.na(wp_make_hat)),
    na_wp_miss    = sum(is.na(wp_miss_hat)),
    na_lev        = sum(is.na(leverage)),
    na_lev_rules  = sum(is.na(leverage_rules))
  )
assert_true(all(pats_na[1, c("na_wp_make","na_wp_miss","na_lev","na_lev_rules")] == 0),
            "PAT rows contain NAs but should be fully scored (including OT).")

# 4) Rule layer sanity
fg_nonpat_nonot_rules_na <- fg_nonpat_nonot %>% summarise(na_lev_rules = sum(is.na(leverage_rules)))
assert_true(fg_nonpat_nonot_rules_na$na_lev_rules == 0,
            "Non-OT FG rows have NA in leverage_rules unexpectedly.")

# 5) Summaries
summary_counts <- fg_chk %>%
  mutate(kind = if_else(pat_flag, "PAT", "FG"),
         period = if_else(ot_flag, "OT", "Reg")) %>%
  count(kind, period, name = "plays")

na_summary <- fg_chk %>%
  mutate(kind = if_else(pat_flag, "PAT", "FG"),
         period = if_else(ot_flag, "OT", "Reg")) %>%
  summarise(
    plays        = n(),
    wp_make_na   = sum(is.na(wp_make_hat)),
    wp_miss_na   = sum(is.na(wp_miss_hat)),
    lev_na       = sum(is.na(leverage)),
    lev_rules_na = sum(is.na(leverage_rules)),
    .by = c(kind, period)
  ) %>%
  arrange(kind, period)

range_summary <- fg_chk %>%
  summarise(
    across(
      c(wp_make_hat, wp_miss_hat, leverage, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules),
      list(min = ~min(.x, na.rm = TRUE), max = ~max(.x, na.rm = TRUE)),
      .names = "{.col}_{.fn}"
    )
  )

cat("\n=== Sanity Check Report =====================================\n")
summary_counts
cat("\n--- NA counts by group ---------------------------------------\n")
na_summary
cat("\n--- Value ranges (excluding NAs) ------------------------------\n")
range_summary

# 6) Edge-case spot checks
edge_hi <- fg_chk %>%
  filter(!is.na(leverage_rules)) %>%
  arrange(desc(leverage_rules)) %>%
  select(game_id, play_id, is_pat, is_ot, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules) %>%
  head(5)

edge_lo <- fg_chk %>%
  filter(!is.na(leverage_rules)) %>%
  arrange(leverage_rules) %>%
  select(game_id, play_id, is_pat, is_ot, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules) %>%
  head(5)

cat("\n--- Top 5 highest leverage_rules ------------------------------\n"); edge_hi
cat("\n--- Top 5 lowest leverage_rules -------------------------------\n"); edge_lo
cat("\nSanity checks completed.\n")



=== Sanity Check Report =====================================


kind,period,plays
<chr>,<chr>,<int>
FG,OT,138
FG,Reg,10521
PAT,Reg,12749



--- NA counts by group ---------------------------------------


kind,period,plays,wp_make_na,wp_miss_na,lev_na,lev_rules_na
<chr>,<chr>,<int>,<int>,<int>,<int>,<int>
FG,OT,138,138,138,0,0
FG,Reg,10521,0,0,0,0
PAT,Reg,12749,0,0,0,0



--- Value ranges (excluding NAs) ------------------------------


...


--- Top 5 highest leverage_rules ------------------------------


game_id,play_id,is_pat,is_ot,wp_make_hat_rules,wp_miss_hat_rules,leverage_rules
<chr>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,<dbl>
2015_05_CHI_KC,4185,0,FALSE,0.995,0.005,0.99
2015_10_DET_GB,4687,0,FALSE,0.995,0.005,0.99
2015_10_JAX_BAL,4237,0,FALSE,0.995,0.005,0.99
2016_01_OAK_NO,4667,0,FALSE,0.995,0.005,0.99
2016_15_TEN_KC,3778,0,FALSE,0.995,0.005,0.99



--- Top 5 lowest leverage_rules -------------------------------


game_id,play_id,is_pat,is_ot,wp_make_hat_rules,wp_miss_hat_rules,leverage_rules
<chr>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,<dbl>
2015_03_JAX_NE,3872,1,FALSE,0.9999963,0.9999964,0
2015_03_SF_ARI,3469,1,FALSE,0.9999846,0.9999846,0
2015_05_NO_PHI,4458,1,FALSE,0.0000010,0.0000010,0
2015_16_NYG_MIN,3768,1,FALSE,0.9999927,0.9999928,0
2016_04_KC_PIT,4301,1,FALSE,0.0000010,0.0000010,0



Sanity checks completed.


## Standardized Features & Modeling Flags


In [72]:
stats_attempts <- fg_all %>%
  filter(attempted == 1L) %>%
  summarise(
    wind_mean = mean(wind, na.rm = TRUE),
    wind_sd = sd(wind, na.rm = TRUE),
    temp_mean = mean(temp, na.rm = TRUE),
    temp_sd = sd(temp, na.rm = TRUE),
    humidity_mean = mean(humidity, na.rm = TRUE),
    humidity_sd = sd(humidity, na.rm = TRUE),
    age_mean = mean(kicker_age, na.rm = TRUE),
    age_sd = sd(kicker_age, na.rm = TRUE),
    exp_mean = mean(kicker_experience, na.rm = TRUE),
    exp_sd = sd(kicker_experience, na.rm = TRUE),
    season_mean = mean(season, na.rm = TRUE),
    season_sd = sd(season, na.rm = TRUE)
  )

wind_mean <- stats_attempts$wind_mean
wind_sd <- stats_attempts$wind_sd
if (is.na(wind_sd) || wind_sd == 0) wind_sd <- NA_real_

temp_mean <- stats_attempts$temp_mean
temp_sd <- stats_attempts$temp_sd
if (is.na(temp_sd) || temp_sd == 0) temp_sd <- NA_real_

humidity_mean <- stats_attempts$humidity_mean
humidity_sd <- stats_attempts$humidity_sd
if (is.na(humidity_sd) || humidity_sd == 0) humidity_sd <- NA_real_

age_mean <- stats_attempts$age_mean
age_sd <- stats_attempts$age_sd
if (is.na(age_sd) || age_sd == 0) age_sd <- NA_real_

exp_mean <- stats_attempts$exp_mean
exp_sd <- stats_attempts$exp_sd
if (is.na(exp_sd) || exp_sd == 0) exp_sd <- NA_real_

season_mean <- stats_attempts$season_mean
season_sd <- stats_attempts$season_sd
if (is.na(season_sd) || season_sd == 0) season_sd <- NA_real_

# Precompute global mean/sd for leverage_rules (exclude NA)
lev_mean <- mean(fg_all$leverage_rules, na.rm = TRUE)
lev_sd   <- stats::sd(fg_all$leverage_rules, na.rm = TRUE)

fg_all <- fg_all %>%
  mutate(
    wind_z = ifelse(!is.na(wind) & !is.na(wind_sd), (wind - wind_mean) / wind_sd, NA_real_),
    temp_z = ifelse(!is.na(temp) & !is.na(temp_sd), (temp - temp_mean) / temp_sd, NA_real_),
    humidity_z = ifelse(!is.na(humidity) & !is.na(humidity_sd), (humidity - humidity_mean) / humidity_sd, NA_real_),
    kicker_age_z = ifelse(!is.na(kicker_age) & !is.na(age_sd), (kicker_age - age_mean) / age_sd, NA_real_),
    kicker_experience_z = ifelse(!is.na(kicker_experience) & !is.na(exp_sd), (kicker_experience - exp_mean) / exp_sd, NA_real_),
    season_z = ifelse(!is.na(season) & !is.na(season_sd), (season - season_mean) / season_sd, NA_real_),
    kick_made = ifelse(attempted == 1L & !is.na(kick_result), as.integer(kick_result == "made"), NA_integer_),
    eoh_urgency = as.integer(qtr == 2L & quarter_seconds_remaining <= 10 & clock_running == 1L),
    eog_urgency = as.integer(qtr == 4L & l2m == 1L & clock_running == 1L & !is.na(score_differential) & abs(score_differential) <= 4),
    go_ahead = as.integer(score_differential == 0),
    to_tie = as.integer(score_differential == -3),
    close7 = as.integer(!is.na(score_differential) & abs(score_differential) <= 7),

    # NEW: standardized leverage (global)
    leverage_z = ifelse(!is.na(leverage_rules) & is.finite(lev_sd) & lev_sd > 0,
                        (leverage_rules - lev_mean) / lev_sd, NA_real_)
  )



## Split Data & Basic QA


In [73]:
fg_attempts <- fg_all %>% filter(attempted == 1L) %>% arrange(season, week, game_id, play_id)
fg_nonattempts <- fg_all %>% filter(attempted == 0L) %>% arrange(season, week, game_id, play_id)

fg_attempts %>%
  select(game_id, season, week, kick_result, kick_distance, wind, wind_z, temp, temp_z, kicker_player_name, kick_made) %>%
  head()

fg_nonattempts %>%
  select(game_id, season, week, posteam, defteam, kick_distance, wind, temp, ydstogo, score_differential) %>%
  head()

fg_attempts %>%
  summarise(
    attempts = n(),
    pats = sum(is_pat == 1L, na.rm = TRUE),
    min_distance = min(kick_distance, na.rm = TRUE),
    max_distance = max(kick_distance, na.rm = TRUE),
    missing_temp = sum(is.na(temp)),
    missing_wind = sum(is.na(wind))
  )

fg_nonattempts %>%
  summarise(
    opportunities = n(),
    min_distance = min(kick_distance, na.rm = TRUE),
    max_distance = max(kick_distance, na.rm = TRUE)
  )


game_id,season,week,kick_result,kick_distance,wind,wind_z,temp,temp_z,kicker_player_name,kick_made
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
2015_01_BAL_DEN,2015,1,made,57,13,1.154244,88,1.608162,B.McManus,1
2015_01_BAL_DEN,2015,1,made,56,13,1.154244,88,1.608162,B.McManus,1
2015_01_BAL_DEN,2015,1,made,52,13,1.154244,88,1.608162,J.Tucker,1
2015_01_BAL_DEN,2015,1,made,43,13,1.154244,88,1.608162,B.McManus,1
2015_01_BAL_DEN,2015,1,made,33,13,1.154244,88,1.608162,J.Tucker,1
2015_01_BAL_DEN,2015,1,made,44,13,1.154244,88,1.608162,J.Tucker,1


game_id,season,week,posteam,defteam,kick_distance,wind,temp,ydstogo,score_differential
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2015_03_CIN_BAL,2015,3,CIN,BAL,25,12,70,5,4
2017_05_NYJ_CLE,2017,5,CLE,NYJ,21,8,67,2,-3
2017_09_ATL_CAR,2017,9,CAR,ATL,29,4,61,4,3
2019_08_TB_TEN,2019,8,TEN,TB,45,3,59,2,4
2019_10_KC_TEN,2019,10,KC,TEN,46,9,63,7,5
2019_12_DAL_NE,2019,12,NE,DAL,32,16,38,15,4


attempts,pats,min_distance,max_distance,missing_temp,missing_wind
<int>,<int>,<dbl>,<dbl>,<int>,<int>
23373,12749,18,70,0,0


opportunities,min_distance,max_distance
<int>,<dbl>,<dbl>
35,19,50


## Persist Outputs


In [74]:
attempts_path <- file.path(DATA_DIR, "fg_attempts.csv")
nonattempts_path <- file.path(DATA_DIR, "fg_nonattempts.csv")

readr::write_csv(fg_attempts, attempts_path)
readr::write_csv(fg_nonattempts, nonattempts_path)

list(
  fg_attempts = attempts_path,
  fg_nonattempts = nonattempts_path
)


$fg_attempts
[1] "g:/My files/Python/Sports Analytics Projects/Football/Kickers/Model/Data/fg_attempts.csv"

$fg_nonattempts
[1] "g:/My files/Python/Sports Analytics Projects/Football/Kickers/Model/Data/fg_nonattempts.csv"